# Multi-LLM Deliberation Framework for Research Paper Analysis

**Version** : OpenAI Responses API (OpenAI-Only Council)

**Author** : Shoeb Shakil Sutar

**Institution** : Symbiosis Institute of Technology, Pune

---

## Overview

This notebook implements the **LLM Council** using exclusively OpenAI models via the **OpenAI Responses API**. Unlike the OpenRouter version, this version uses the native OpenAI SDK which provides native PDF support via `file_id` or `file_url`, and two distinct model families — reasoning models (o-series) and language models (GPT-series).

The system works in three stages:
- **Stage 1** : Four council models independently answer the research query in parallel
- **Stage 2** : Responses are anonymized to eliminate self-preference bias
- **Stage 3** : A meta-model evaluates all responses and synthesizes a final answer

---

## Council Models

| Model | Family | Research Strength |
|---|---|---|
| `gpt-5.4-mini` | Language Model | Fast, efficient general QA |
| `o3` | Reasoning Model | Deep multi-step academic reasoning |
| `gpt-5.1` | Language Model | Balanced research analysis |
| `gpt-5.2` | Language Model | Strong academic writing and synthesis |

**Meta-Model (Chairman)** : `gpt-5.4`

---

## Input Modes

| Mode | Description |
|---|---|
| `text` | Plain research query or pasted abstract |
| `url` | Direct link to a PDF (e.g. arXiv URL) |
| `upload` | Upload a local PDF file via OpenAI Files API |

---

## Requirements

- Python 3.10+
- `openai` library
- OpenAI API Key stored as `OpenAI` in Colab secrets
- `Text_Model_Judge_Response.txt` uploaded to `/content/`

---

## Step 1 — Install Dependencies

In [ ]:
!pip install openai -q

## Step 2 — Imports

In [ ]:
from openai import OpenAI
import time
import string
import threading

## Step 3 — API Key

> Add your OpenAI API key to Colab Secrets with the name `OpenAI`.
> Go to : `Tools → Secrets → Add new secret`

In [ ]:
from google.colab import userdata
API_KEY = userdata.get("OpenAI")

## Step 4 — Configuration

In [ ]:
# Council members
MODELS = {
    "llm_1" : "gpt-5.4-mini",
    "llm_2" : "o3",
    "llm_3" : "gpt-5.1",
    "llm_4" : "gpt-5.2"
}

# Chairman
META_MODEL = "gpt-5.4"

# System prompt
SYSTEM_PROMPT = """You are an expert Academic Research Assistant.
When given a research paper, abstract, or research query, analyze it with
academic rigor — focus on methodology, key findings, limitations,
and real-world implications."""

TEMPERATURE = 0.7

## Step 5 — Initialize Client

In [ ]:
client = OpenAI(api_key = API_KEY)

## Step 6 — Core Functions

### 6.1 `file_upload()` — Upload PDF to OpenAI Files API

Uploads a local PDF and returns a `file_id` used in subsequent API calls.

In [ ]:
def file_upload (file_path : str, client = client) -> str :
  file = client.files.create(
      file    = open(file_path, "rb"),
      purpose = "assistants"
  )
  return file.id

### 6.2 `call_llm()` — Single Model Call via Responses API

Handles all three modes — text, url, upload — in a single function.
Returns model name, content, latency, and status.

In [ ]:
def call_llm (model_name : str, prompt : str, client = client,
              file_url = None, file_id = None) -> dict :
  try :
    start_time = time.time()

    # Text only
    if file_url is None and file_id is None :
      response = client.responses.create(
          model = model_name,
          input = [
              {"role" : "system", "content" : SYSTEM_PROMPT},
              {"role" : "user",   "content" : prompt}
          ],
          temperature = TEMPERATURE
      )

    # PDF via URL
    elif file_url is not None :
      response = client.responses.create(
          model = model_name,
          input = [
              {"role" : "system", "content" : SYSTEM_PROMPT},
              {"role" : "user",   "content" : [
                  {"type" : "input_text", "text"     : prompt},
                  {"type" : "input_file", "file_url" : file_url}
              ]}
          ],
          temperature = TEMPERATURE
      )

    # PDF via uploaded file ID
    elif file_id is not None :
      response = client.responses.create(
          model = model_name,
          input = [
              {"role" : "system", "content" : SYSTEM_PROMPT},
              {"role" : "user",   "content" : [
                  {"type" : "input_text", "text"    : prompt},
                  {"type" : "input_file", "file_id" : file_id}
              ]}
          ],
          temperature = TEMPERATURE
      )

    latency = round(time.time() - start_time, 3)

    return {
        "Model"   : model_name,
        "Content" : response.output_text,
        "Latency" : latency,
        "Status"  : "Success"
    }

  except Exception as e :
    return {
        "Model"   : model_name,
        "Content" : None,
        "Latency" : None,
        "Status"  : "Failure",
        "Error"   : str(e)
    }

### 6.3 `query_llms()` — Parallel Council Inference

Dispatches query to all four council models simultaneously using Python threading.
Total latency ≈ slowest model, not sum of all models.

In [ ]:
def query_llms (prompt : str, file_url = None,
                file_id = None) -> dict :
  try :
    responses = {}
    threads   = []

    def worker (key, value, file_url, file_id) :
      responses[key] = call_llm(value, prompt, client,
                                file_url, file_id)

    for key, value in MODELS.items() :
      t = threading.Thread(target = worker,
                           args   = (key, value, file_url, file_id))
      threads.append(t)
      t.start()

    for thread in threads :
      thread.join()

    return responses

  except Exception as e :
    return {
        "Error" : str(e)
    }

### 6.4 `blind_responses()` — Anonymization Layer

Replaces model identities with neutral labels (Response A, B, C, D).
Eliminates self-preference bias in meta-model evaluation.

In [ ]:
def blind_responses (responses : dict) -> dict :
  try :
    letters        = string.ascii_uppercase
    idx            = 0
    blind_response = {}

    for key, value in responses.items() :
      if value["Status"] == "Success" :
        blind_id                 = "Response " + letters[idx]
        blind_response[blind_id] = value["Content"]
        idx = idx + 1

    return blind_response

  except Exception as e :
    return {
        "Error" : str(e)
    }

### 6.5 `final_judge_response()` — Deliberation Layer

Sends query and anonymized responses to meta-model for evaluation and synthesis.
PDF context is also passed if available, allowing the meta-model to verify responses against the source paper.

Scores each response on Correctness, Clarity, Completeness, Academic Depth — each /10, total /40.

In [ ]:
def final_judge_response (prompt : str, blind_response : dict,
                          file_url = None, file_id = None) -> str :
  try :
    with open("/content/Text_Model_Judge_Response.txt") as file :
      instruction = file.read()

    final_judge_prompt = instruction.format(
        user_prompt     = prompt,
        blind_responses = blind_response
    )

    result = call_llm(META_MODEL, final_judge_prompt, client,
                      file_url, file_id)
    return result["Content"]

  except Exception as e :
    return str(e)

### 6.6 `run_llm_council()` — Master Orchestration Function

Runs the full three-stage pipeline end-to-end.

In [ ]:
def run_llm_council () :
  try :
    print("=== Research Paper Assistant (LLM Council) ===")
    print("Modes : text | url | upload")
    mode   = input("Enter Preferred Mode : ")
    prompt = input("Enter Your Question : ")

    # Stage 1 : Query council models
    if mode.lower() == "text" :
      responses    = query_llms(prompt)
      blind        = blind_responses(responses)
      final_answer = final_judge_response(prompt, blind)

    elif mode.lower() == "url" :
      file_url     = input("Enter PDF URL : ")
      responses    = query_llms(prompt, file_url = file_url)
      blind        = blind_responses(responses)
      final_answer = final_judge_response(prompt, blind,
                                          file_url = file_url)

    elif mode.lower() == "upload" :
      file_path = input("Enter Path of File : ")
      file_id   = file_upload(file_path, client)
      print(f"File Uploaded Successfully | ID : {file_id}")
      responses    = query_llms(prompt, file_id = file_id)
      blind        = blind_responses(responses)
      final_answer = final_judge_response(prompt, blind,
                                          file_id = file_id)

    else :
      print("Invalid Mode. Please enter text, url, or upload.")
      return

    # Display latency summary
    print("\n--- Council Responses ---")
    for key, val in responses.items() :
      print(f"  {key} ({val['Model']}) → {val['Status']} | Latency : {val.get('Latency', 'N/A')}s")

    # Display final answer
    print("\n--- Final Response (Meta Model) ---")
    print(final_answer)

  except Exception as e :
    print("Error : ", str(e))

## Step 7 — Run the LLM Council

> **Before running :**
> - Upload `Text_Model_Judge_Response.txt` to `/content/`
> - For `upload` mode, upload your PDF to `/content/` and note the file path

In [ ]:
run_llm_council()

---

## Differences from OpenRouter Version

| Feature | OpenRouter Version | OpenAI Version |
|---|---|---|
| API | Chat Completions | Responses API |
| SDK call | `client.chat.completions.create()` | `client.responses.create()` |
| Response field | `response.choices[0].message.content` | `response.output_text` |
| Models | Multi-provider | OpenAI only |
| PDF Support | Not supported | Native via `file_id` / `file_url` |
| File Upload | Not supported | OpenAI Files API |

---

## References

1. Vaswani et al. (2017). *Attention is All You Need*. NeurIPS.
2. Karpathy, A. (2025). *LLM Council*. https://github.com/karpathy/llm-council
3. Zheng et al. (2023). *Judging LLM-as-a-Judge with MT-Bench*. arXiv:2306.05685.
4. OpenAI. (2025). *Responses API*. https://platform.openai.com/docs/api-reference/responses
5. OpenAI. (2025). *Files API*. https://platform.openai.com/docs/api-reference/files
6. Dietterich, T. G. (2000). *Ensemble Methods in Machine Learning*. Springer.

---

*Symbiosis Institute of Technology, Pune — MTech AI & ML — 2025-2026*